# Sinmate: QLoRA Fine-Tuning Gemma-3-1B-IT for Spoken Sinhala (Jupyter / Global)

This notebook fine-tunes **`google/gemma-3-1b-it`** using **4-bit QLoRA** on a conversational Sinhala dataset (*Kathitha Bhashawa* with Sri Lankan fillers, Singlish, and 4 personas).

It is the **Jupyter-server twin** of `notebooks/colab/02_colab_qlora_finetune_gemma3.ipynb`: same cell numbers and order, but the Colab-only parts (Colab secrets, `!` shell magics, `/content` paths, the apt-based Ollama install) are replaced with portable code that runs on a regular Jupyter server (local workstation, remote GPU box, JupyterHub). Keep the two notebooks in sync.

### Requirements:
- **NVIDIA GPU + CUDA build of PyTorch**: 4-bit QLoRA (`bitsandbytes`) does not run on CPU-only machines. The free Colab T4 (15GB) is known to work.
- `git`, and `cmake` + a C++ compiler to build `llama-quantize` in Cell 9.
- Optional: [Ollama](https://ollama.com/download) for Cells 11-12 (Cell 12 falls back to HuggingFace Transformers without it).
- A Hugging Face token with Gemma access (`HF_TOKEN` env var, cached `huggingface-cli login`, or paste when prompted).

### Pipeline Overview:
1. **Load Base Model**: `google/gemma-3-1b-it` quantized to 4-bit (VRAM footprint ~1.8GB).
2. **Chat Template**: Standard Gemma turn tags `<start_of_turn>user...<end_of_turn>`.
3. **Train with QLoRA**: LoRA adapters on linear attention and MLP layers (`r=16, alpha=32`).
4. **Inference Test**: Multi-persona conversational test in colloquial Sinhala.
5. **Export GGUF**: Export 4-bit quantized GGUF weights (~700MB–800MB) for offline laptop execution.

In [2]:
# 1. Install fine-tuning dependencies into this kernel's environment
%pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece "torchao>=0.16.0"

# 4-bit QLoRA needs an NVIDIA CUDA build of PyTorch (https://pytorch.org/get-started/locally/)
import torch
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

Note: you may need to restart the kernel to use updated packages.
torch 2.6.0+cu124 | CUDA available: True


In [3]:
# 2. Hugging Face Login (Access Gemma license)
# Token lookup order: HF_TOKEN env var -> cached `huggingface-cli login` -> getpass prompt
import os
from getpass import getpass
from huggingface_hub import get_token, login

token = os.environ.get('HF_TOKEN') or get_token() or getpass('Enter your Hugging Face access token: ')
login(token=token)

In [5]:
# 3. Force a fresh clone of Sinmate and load the dataset

import os
import shutil
import subprocess
from pathlib import Path
from datasets import load_dataset

REPO_URL = "https://github.com/chathuwa-whiz/Sinmate.git"
FORCE_RECLONE = True

# Find the workspace, even if the current directory is inside Sinmate
cwd = Path.cwd().resolve()
existing_repo = next(
    (p for p in [cwd, *cwd.parents] if p.name == "Sinmate"),
    None
)

workspace = existing_repo.parent if existing_repo else cwd
os.chdir(workspace)

repo_root = workspace / "Sinmate"

# Remove the existing folder if a fresh clone is requested
if FORCE_RECLONE and repo_root.exists():
    print(f"Removing existing repository: {repo_root}")
    shutil.rmtree(repo_root)

# Clone the repository
if not repo_root.exists():
    print("Cloning Sinmate from GitHub...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(repo_root)],
        check=True
    )

# Change to the repository directory
os.chdir(repo_root)
print("Working directory:", repo_root)

# Verify that the dataset files exist
train_file = repo_root / "data" / "train.jsonl"
val_file = repo_root / "data" / "val.jsonl"

if not train_file.exists() or not val_file.exists():
    raise FileNotFoundError(
        "Dataset files are missing. Expected data/train.jsonl "
        "and data/val.jsonl inside the repository."
    )

# Load the dataset
dataset = load_dataset(
    "json",
    data_files={
        "train": str(train_file),
        "val": str(val_file),
    }
)

print("Dataset loaded successfully:")
print(dataset)

Removing existing repository: /home/jovyan/Sinmate
Cloning Sinmate from GitHub...


Cloning into '/home/jovyan/Sinmate'...


Working directory: /home/jovyan/Sinmate


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['id', 'persona', 'input_type', 'emotion', 'scenario', 'messages', 'category'],
        num_rows: 106
    })
    val: Dataset({
        features: ['id', 'persona', 'input_type', 'emotion', 'scenario', 'messages', 'category'],
        num_rows: 11
    })
})


In [6]:
# 4. Load Gemma-3-1B-IT in 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-3-1b-it"

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected: 4-bit QLoRA (bitsandbytes) needs an NVIDIA GPU. "
        "Use a CUDA-enabled Jupyter server, or the Colab notebook in notebooks/colab/."
    )

# bf16 needs compute capability >= 8.0 (A100/L4/...). The free-tier T4 is 7.5: bf16 kernels fail there,
# and Gemma 3 overflows in fp16, so fall back to fp32 compute (fine for a 1B model).
USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
print(f"GPU: {torch.cuda.get_device_name()} | compute dtype: {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.6.0+cu124).
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.6.0+cu124).


ImportError: cannot import name 'BloomPreTrainedModel' from 'transformers' (/opt/conda/lib/python3.11/site-packages/transformers/__init__.py)

In [ ]:
# 5. Format Chat Templates & Train with Response-Only Loss Masking
from trl import SFTConfig, SFTTrainer

# Ensure tokenizer padding token is defined for batched training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Use conversational messages format directly (TRL applies chat template automatically)
train_dataset = dataset["train"].select_columns(["messages"])
val_dataset = dataset["val"].select_columns(["messages"])

# TRL v0.14+ natively supports assistant_only_loss in SFTConfig for conversational datasets.
# In TRL >= 0.20, DataCollatorForCompletionOnlyLM has been deprecated and removed.
try:
    training_args = SFTConfig(
        output_dir="./gemma3_sinhala_output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=0.05,  # float < 1 = ratio of total steps
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=10,
        num_train_epochs=3,
        fp16=False,
        bf16=USE_BF16,  # set in cell 4; False on T4 (fp32 compute)
        save_strategy="epoch",
        eval_strategy="epoch",
        max_length=512,
        assistant_only_loss=True,  # Response-only loss: computes loss ONLY on assistant turns
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=training_args,
    )
except (TypeError, ValueError):
    # Fallback for legacy TRL versions (< 0.14) that required DataCollatorForCompletionOnlyLM
    from trl import DataCollatorForCompletionOnlyLM

    def format_prompts(example):
        return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

    train_dataset = dataset["train"].map(format_prompts, remove_columns=dataset["train"].column_names)
    val_dataset = dataset["val"].map(format_prompts, remove_columns=dataset["val"].column_names)

    collator = DataCollatorForCompletionOnlyLM(
        response_template="<start_of_turn>model\n",
        instruction_template="<start_of_turn>user\n",
        tokenizer=tokenizer,
    )

    training_args = SFTConfig(
        output_dir="./gemma3_sinhala_output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=0.05,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=10,
        num_train_epochs=3,
        fp16=False,
        bf16=USE_BF16,
        save_strategy="epoch",
        eval_strategy="epoch",
        dataset_text_field="text",
        max_length=512,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collator,
        processing_class=tokenizer,
        args=training_args,
    )

trainer.train()


In [ ]:
# 6. Test Interactive Multi-Persona Inference
test_messages = [
    {
        "role": "system",
        "content": "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."
    },
    {"role": "user", "content": "Mata ada mara kammali ban, monada karanna hoda?"}
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7, top_p=0.9, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n🤖 SINMATE RESPONSE:", response)


In [ ]:
# 7. Save LoRA Adapters
model.save_pretrained("sinmate-gemma-3-1b-lora")
tokenizer.save_pretrained("sinmate-gemma-3-1b-lora")
print("Adapters saved successfully to sinmate-gemma-3-1b-lora!")

In [ ]:
# 8. Merge LoRA with Base Model for GGUF Export
import sys
import gc
import json
import torch

# Fix Colab incompatibility where an outdated torchao (< 0.16.0) is pre-installed.
# Bypasses torchao dispatcher check in PEFT so standard PyTorch linear LoRA merge succeeds.
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    if hasattr(peft.import_utils, "TORCHAO_MINIMUM_VERSION"):
        peft.import_utils.TORCHAO_MINIMUM_VERSION = "0.0.0"
    for mod_name, mod in list(sys.modules.items()):
        if mod_name.startswith("peft") and hasattr(mod, "is_torchao_available"):
            setattr(mod, "is_torchao_available", lambda: False)
except Exception:
    pass

from peft import PeftModel
from transformers import AutoModelForCausalLM

# Clear CUDA cache before merging on CPU to avoid memory pressure
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading base model in FP16 to merge LoRA...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
)
merged_model = PeftModel.from_pretrained(base_model, "sinmate-gemma-3-1b-lora")
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("sinmate-gemma-3-1b-merged")
tokenizer.save_pretrained("sinmate-gemma-3-1b-merged")

# Align vocab_size in config.json with actual tokenizer vocabulary to satisfy llama.cpp GGUF converter
config_path = "sinmate-gemma-3-1b-merged/config.json"
try:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    max_tok_id = max(tokenizer.get_vocab().values())
    if cfg.get("vocab_size", 0) <= max_tok_id:
        cfg["vocab_size"] = max_tok_id + 1
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"Aligned config.json vocab_size -> {cfg['vocab_size']} (max token ID: {max_tok_id})")
except Exception as e:
    print(f"Note: vocab_size alignment: {e}")

print("Merged model saved successfully to ./sinmate-gemma-3-1b-merged!")


In [ ]:
# 9. Convert Merged Model to 4-bit GGUF via llama.cpp
import os
import sys
import json
import shutil
import subprocess

def run(cmd, check=False):
    """Run a command and stream its output into the cell (a bare subprocess would print to the Jupyter server's terminal)."""
    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")
    return proc.returncode

# Helper to locate directories regardless of the notebook's working directory
def resolve_path(target_name):
    candidates = [
        target_name,
        os.path.join("Sinmate", target_name),
        os.path.join("..", target_name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

# 1. Locate merged model directory
merged_model_dir = resolve_path("sinmate-gemma-3-1b-merged")
if not merged_model_dir:
    raise FileNotFoundError("Could not find 'sinmate-gemma-3-1b-merged'. Please ensure Cell 8 completed successfully.")
print(f"Found merged model directory: {merged_model_dir}")

# 2. Clone llama.cpp and install conversion requirements (avoiding legacy protobuf downgrade)
llama_cpp_dir = resolve_path("llama.cpp")
if not llama_cpp_dir:
    run(["git", "clone", "https://github.com/ggml-org/llama.cpp.git"], check=True)
    llama_cpp_dir = resolve_path("llama.cpp") or os.path.abspath("llama.cpp")
print(f"Found llama.cpp directory: {llama_cpp_dir}")

# Install only what the converter needs. llama.cpp's own requirements files can pull in a different (CPU-only)
# torch build, which would silently replace this environment's CUDA torch.
pip_install = [sys.executable, "-m", "pip", "install", "-q"]
run(pip_install + [os.path.join(llama_cpp_dir, "gguf-py"), "sentencepiece", "--upgrade", "protobuf>=5.0.0"], check=True)

# 3. Gemma 3 must be converted from its SentencePiece `tokenizer.model`. Without it llama.cpp falls back to the
#    GPT-2/BPE route, mis-detects the pre-tokenizer and writes 262145 tokens (incl. <image_soft_token>) against
#    262144 embedding rows - a GGUF that Ollama cannot load (HTTP 500 on every chat request).
tok_model_path = os.path.join(merged_model_dir, "tokenizer.model")
if not os.path.exists(tok_model_path):
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download("google/gemma-3-1b-it", "tokenizer.model", local_dir=merged_model_dir)
    except Exception as e:
        print(f"⚠️ Could not fetch tokenizer.model from google/gemma-3-1b-it: {e}")
has_spm = os.path.exists(tok_model_path)
print(f"SentencePiece tokenizer.model present: {has_spm}")

# 4. config.json vocab_size must equal the embedding rows of the weights. Earlier runs bumped it to 262145 (the
#    tokenizer's extra <image_soft_token>) and that value persists on disk: the converter then pads the vocab to
#    262145 tokens against 262144 embedding rows. Reset it from the actual weights.
if has_spm:
    import glob
    from safetensors import safe_open
    config_path = os.path.join(merged_model_dir, "config.json")
    embd_rows = None
    for st_file in sorted(glob.glob(os.path.join(merged_model_dir, "*.safetensors"))):
        with safe_open(st_file, framework="pt") as f:
            for key in f.keys():
                if key.endswith("embed_tokens.weight"):
                    embd_rows = f.get_slice(key).get_shape()[0]
                    break
        if embd_rows:
            break
    if embd_rows and os.path.exists(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        if cfg.get("vocab_size") != embd_rows:
            print(f"Reset config.json vocab_size {cfg.get('vocab_size')} -> {embd_rows} (embedding rows)")
            cfg["vocab_size"] = embd_rows
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(cfg, f, indent=2)
        else:
            print(f"config.json vocab_size already matches embedding rows ({embd_rows})")

# 5. Only without tokenizer.model: fall back to the vocab_size / assertion workarounds for the BPE route
if not has_spm:
    config_path = os.path.join(merged_model_dir, "config.json")
    if os.path.exists(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(merged_model_dir)
        max_tok_id = max(tok.get_vocab().values())
        if cfg.get("vocab_size", 0) <= max_tok_id:
            cfg["vocab_size"] = max_tok_id + 1
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(cfg, f, indent=2)
            print(f"Aligned config.json vocab_size -> {cfg['vocab_size']} (max token ID: {max_tok_id})")
    # Patch conversion/base.py in llama.cpp to prevent assertion crashes
    base_py = os.path.join(llama_cpp_dir, "conversion", "base.py")
    if os.path.exists(base_py):
        with open(base_py, "r", encoding="utf-8") as f:
            content = f.read()
        if "assert max(tokenizer.vocab.values()) < vocab_size" in content:
            content = content.replace(
                "assert max(tokenizer.vocab.values()) < vocab_size",
                "vocab_size = max(vocab_size, max(tokenizer.vocab.values()) + 1)"
            )
            with open(base_py, "w", encoding="utf-8") as f:
                f.write(content)
            print("Patched llama.cpp conversion/base.py assertion check")

# 6. Convert HuggingFace checkpoint to FP16 GGUF
f16_gguf = os.path.abspath("sinmate-gemma-3-1b-f16.gguf")
q4_gguf = os.path.abspath("sinmate-gemma-3-1b-q4_k_m.gguf")
# Remove GGUFs from earlier runs so a failed conversion can never leave a stale file in use
for stale in (f16_gguf, q4_gguf):
    if os.path.exists(stale):
        os.remove(stale)
convert_script = os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")
run([sys.executable, convert_script, merged_model_dir, "--outfile", f16_gguf, "--outtype", "f16"])

# 7. Locate or compile llama-quantize with memory-safe parallelism (-j 2)
exe = ".exe" if os.name == "nt" else ""

def find_quantize():
    for bin_candidate in [
        os.path.join(llama_cpp_dir, "build", "bin", "Release", "llama-quantize" + exe),  # MSVC multi-config layout
        os.path.join(llama_cpp_dir, "build", "bin", "llama-quantize" + exe),
        os.path.join(llama_cpp_dir, "build", "llama-quantize" + exe),
        os.path.join(llama_cpp_dir, "llama-quantize" + exe),
    ]:
        if os.path.exists(bin_candidate):
            return os.path.abspath(bin_candidate)
    return None

quantize_bin = find_quantize()

if not quantize_bin:
    if shutil.which("cmake") is None:
        raise RuntimeError(
            "cmake (and a C++ compiler) is required to build llama-quantize. Install them "
            "(e.g. `sudo apt install cmake build-essential`, `brew install cmake`, or Visual Studio Build Tools + cmake) "
            "and re-run this cell."
        )
    build_dir = os.path.join(llama_cpp_dir, "build")
    run(["cmake", "-B", build_dir, llama_cpp_dir, "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_SERVER=OFF", "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF"], check=True)
    run(["cmake", "--build", build_dir, "--config", "Release", "-j", "2", "--target", "llama-quantize"], check=True)
    quantize_bin = find_quantize()
    if not quantize_bin:
        raise RuntimeError("llama-quantize was built but its binary could not be found under " + build_dir)

print(f"Using llama-quantize binary: {quantize_bin}")

# 8. Quantize to Q4_K_M (4-bit)
if os.path.exists(f16_gguf):
    if os.name != "nt":
        os.chmod(quantize_bin, 0o755)
    run([quantize_bin, f16_gguf, q4_gguf, "Q4_K_M"])
else:
    print(f"⚠️ {f16_gguf} was not found. Please review convert_hf_to_gguf logs above.")

if os.path.exists(q4_gguf):
    size_mb = os.path.getsize(q4_gguf) / (1024 * 1024)
    print(f"\n🎉 GGUF quantization complete: {q4_gguf} ({size_mb:.1f} MB)")
else:
    print("\n⚠️ Quantization output file not found. Please check logs above.")


In [ ]:
# 10. Generate Modelfile for Ollama
import os

def resolve_path(target_name):
    candidates = [
        target_name,
        os.path.join("Sinmate", target_name),
        os.path.join("..", target_name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

# Locate the quantized GGUF model produced by Cell 9
gguf_path = resolve_path("sinmate-gemma-3-1b-q4_k_m.gguf") or os.path.abspath("sinmate-gemma-3-1b-q4_k_m.gguf")
print(f"Target GGUF file: {gguf_path}")

# Gemma has no system role: the HF chat template used for fine-tuning folds the system prompt
# into the first user turn, so the Ollama TEMPLATE must do the same to match training.
modelfile_template = """FROM {GGUF_PATH}
TEMPLATE \"\"\"<start_of_turn>user
{{ if .System }}{{ .System }}

{{ end }}{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
\"\"\"
PARAMETER stop "<start_of_turn>"
PARAMETER stop "<end_of_turn>"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
"""

modelfile = modelfile_template.replace("{GGUF_PATH}", gguf_path)

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)

print("\n✅ Modelfile successfully generated at:", os.path.abspath("Modelfile"))
print(f"Points to GGUF: {gguf_path}")
print("\n👉 Next Step: Run Cell 11 to install Ollama, start the daemon, and build 'sinmate-gemma3:1b' on this machine!")

In [ ]:
# 11. Setup Ollama & Build Sinmate Model on This Machine
import os
import sys
import gc
import json
import time
import shutil
import subprocess
import urllib.error
import urllib.request

import torch

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL_TAG = "sinmate-gemma3:1b"

def run(cmd, check=False):
    """Run a command and stream its output into the cell (a bare subprocess would print to the Jupyter server's terminal)."""
    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")
    return proc.returncode

def resolve_path(target_name):
    candidates = [
        target_name,
        os.path.join("Sinmate", target_name),
        os.path.join("..", target_name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

def ollama_up():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False

def ollama_error(e, log_lines=25):
    """Readable reason for a failed Ollama request: the response body (urllib hides it) + the server log tail."""
    body = ""
    if isinstance(e, urllib.error.HTTPError):
        try:
            body = e.read().decode("utf-8", "replace")
        except Exception:
            pass
    tail = ""
    if os.path.exists("ollama_server.log"):
        with open("ollama_server.log", "r", errors="replace") as f:
            tail = "".join(f.readlines()[-log_lines:])
    return f"{e}\n{body}\n--- ollama_server.log (tail) ---\n{tail}"

def check_gguf_vocab(path):
    """Fail fast if the GGUF has more tokens than embedding rows - Ollama then answers every chat with HTTP 500."""
    try:
        from gguf import GGUFReader
        reader = GGUFReader(path)
        n_tokens = len(reader.fields["tokenizer.ggml.tokens"].data)
        embd_rows = next(int(t.shape[1]) for t in reader.tensors if t.name == "token_embd.weight")
    except (ImportError, KeyError, StopIteration) as e:
        print(f"ℹ️ Skipping GGUF vocab check ({type(e).__name__}: {e})")
        return
    print(f"GGUF vocab check: {n_tokens} tokens vs {embd_rows} embedding rows")
    if n_tokens != embd_rows:
        raise RuntimeError(
            f"❌ GGUF is broken: {n_tokens} tokens but {embd_rows} embedding rows. "
            "Re-run Cell 9 (it now converts with Gemma's SentencePiece tokenizer.model), then Cell 10 and this cell."
        )

# 1. Clean up unused PyTorch tensors to leave maximum free VRAM for Ollama
for var_name in ["model", "trainer", "base_model", "merged_model"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Ollama must be installed on the machine running this kernel. Only a root Linux shell can be auto-installed
#    (its installer extracts a .tar.zst, so zstd must exist first); macOS/Windows need the desktop installer.
if shutil.which("ollama") is None:
    if sys.platform.startswith("linux") and hasattr(os, "geteuid") and os.geteuid() == 0:
        print("📦 Installing zstd + Ollama (running as root)...")
        if shutil.which("apt-get"):
            run(["apt-get", "install", "-y", "-qq", "zstd", "pciutils"])
        run(["sh", "-c", "curl -fsSL https://ollama.com/install.sh | sh"])
    if shutil.which("ollama") is None:
        raise RuntimeError(
            "❌ Ollama not found. Install it from https://ollama.com/download (on Linux: "
            "`curl -fsSL https://ollama.com/install.sh | sh`), restart the Jupyter server if `ollama` is still not on PATH, "
            "then re-run this cell. To chat without Ollama, skip to Cell 12 (HuggingFace fallback)."
        )
else:
    print("✅ Ollama binary is already installed.")

# 3. Start background Ollama daemon if not running
if ollama_up():
    print("✅ Ollama server is already running.")
else:
    print("🚀 Starting Ollama daemon in background...")
    log_fp = open("ollama_server.log", "a")
    detach = (
        {"creationflags": subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP}
        if os.name == "nt"
        else {"start_new_session": True}
    )
    subprocess.Popen(["ollama", "serve"], stdout=log_fp, stderr=subprocess.STDOUT, **detach)
    for attempt in range(30):
        if ollama_up():
            print(f"✅ Ollama server is live at {OLLAMA_URL} (attempt {attempt + 1})")
            break
        time.sleep(1)
    else:
        with open("ollama_server.log", "r") as f:
            print("".join(f.readlines()[-20:]))
        raise RuntimeError("❌ Ollama server did not respond - see log output above.")

# 4. Locate Modelfile and GGUF
modelfile_path = resolve_path("Modelfile")
gguf_path = resolve_path("sinmate-gemma-3-1b-q4_k_m.gguf")

if not modelfile_path:
    raise FileNotFoundError("❌ 'Modelfile' not found. Please run Cell 10 first!")
if not gguf_path:
    raise FileNotFoundError("❌ 'sinmate-gemma-3-1b-q4_k_m.gguf' not found. Please run Cell 9 first!")

print(f"Modelfile: {modelfile_path}")
print(f"GGUF: {gguf_path}")
check_gguf_vocab(gguf_path)

# Ensure Modelfile points to the exact absolute GGUF path
with open(modelfile_path, "r", encoding="utf-8") as f:
    mf_lines = f.read().splitlines()

mf_lines = [f"FROM {gguf_path}" if line.startswith("FROM ") else line for line in mf_lines]

with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write("\n".join(mf_lines) + "\n")

# 5. Build the Sinmate model in Ollama
print(f"\n🔨 Building Ollama model '{MODEL_TAG}' from Modelfile...")
run(["ollama", "create", MODEL_TAG, "-f", modelfile_path], check=True)

# 6. List Ollama models
print("\n📋 Available Ollama models:")
run(["ollama", "list"])

# 7. Quick verification through the HTTP API (the `ollama run` CLI prints spinner escape codes into the cell output)
print("\n🧪 Running quick verification prompt via Ollama API...")
verify_payload = json.dumps({
    "model": MODEL_TAG,
    "stream": False,
    "messages": [
        {"role": "system", "content": "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."},
        {"role": "user", "content": "Mata ada mara kammali ban, monada karanna hoda?"},
    ],
    "options": {"temperature": 0.7, "top_p": 0.9, "num_predict": 64},
}).encode("utf-8")
verify_req = urllib.request.Request(OLLAMA_URL + "/api/chat", data=verify_payload, headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(verify_req, timeout=300) as resp:
        print("🤖 SINMATE RESPONSE:", json.loads(resp.read().decode("utf-8"))["message"]["content"])
except Exception as e:
    print(f"⚠️ Verification request failed: {ollama_error(e)}")

In [ ]:
# 12. Real-Time Interactive Multi-Persona Chat
import os
import sys
import json
import urllib.error
import urllib.request

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL_TAG = "sinmate-gemma3:1b"
MAX_HISTORY = 6  # last N messages kept as conversation context

# 1. Define Persona Prompts (Kathitha Bhashawa / Sri Lankan Spoken Sinhala)
PERSONAS = {
    "1": ("BEST_FRIEND", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."),
    "2": ("ROMANTIC_PARTNER", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: ROMANTIC_PARTNER]\nYou are the user's loving, affectionate romantic partner. Responses must strictly be 1-2 warm, spoken colloquial sentences."),
    "3": ("COUNSELLOR", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: COUNSELLOR]\nYou are a calm, deeply empathetic listener. Responses must strictly be 1-2 grounded, supportive spoken colloquial sentences."),
    "4": ("PLAYFUL_SIBLING", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: PLAYFUL_SIBLING]\nYou are the user's witty, teasing sibling. Responses must strictly be 1-2 playful, humorous spoken colloquial sentences."),
}

# 2. Check if the Ollama engine is up and has the Sinmate model
def is_ollama_ready():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=2) as r:
            data = json.loads(r.read().decode("utf-8"))
            return any("sinmate-gemma3" in m.get("name", "") for m in data.get("models", []))
    except Exception:
        return False

ollama_available = is_ollama_ready()

def ollama_error(e, log_lines=25):
    """Readable reason for a failed Ollama request: the response body (urllib hides it) + the server log tail."""
    body = ""
    if isinstance(e, urllib.error.HTTPError):
        try:
            body = e.read().decode("utf-8", "replace")
        except Exception:
            pass
    tail = ""
    if os.path.exists("ollama_server.log"):
        with open("ollama_server.log", "r", errors="replace") as f:
            tail = "".join(f.readlines()[-log_lines:])
    return f"{e}\n{body}\n--- ollama_server.log (tail) ---\n{tail}"

def build_messages(user_text, persona_key, history):
    _, system_prompt = PERSONAS.get(persona_key, PERSONAS["1"])
    messages = [{"role": "system", "content": system_prompt}]
    if history:
        messages.extend(history[-MAX_HISTORY:])
    messages.append({"role": "user", "content": user_text})
    return messages

# 3. Streaming chat functions (both return the full reply so it can be kept as history)
def chat_via_ollama(user_text, persona_key="1", history=None):
    persona_name = PERSONAS.get(persona_key, PERSONAS["1"])[0]
    payload = json.dumps({
        "model": MODEL_TAG,
        "messages": build_messages(user_text, persona_key, history),
        "stream": True,
        "options": {
            "temperature": 0.7,
            "top_p": 0.9,
            "num_predict": 128,
            "stop": ["<start_of_turn>", "<end_of_turn>"],
        },
    }).encode("utf-8")

    req = urllib.request.Request(
        OLLAMA_URL + "/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"},
    )

    print(f"\n[{persona_name}] Sinmate: ", end="")
    full_response = []
    with urllib.request.urlopen(req, timeout=300) as resp:
        for line in resp:
            if not line.strip():
                continue
            token = json.loads(line.decode("utf-8")).get("message", {}).get("content", "")
            if token:
                sys.stdout.write(token)
                sys.stdout.flush()
                full_response.append(token)
    print()
    return "".join(full_response)

# Hugging Face fallback if Ollama is not running
hf_model = None
hf_tokenizer = None

def init_hf_fallback():
    global hf_model, hf_tokenizer
    if hf_model is not None:
        return

    os.environ["TRANSFORMERS_NO_TF"] = "1"
    os.environ["USE_TF"] = "0"
    try:
        import transformers.utils.import_utils as tr_import
        tr_import.is_tf_available = lambda: False
        tr_import.is_flax_available = lambda: False
        tr_import.is_torch_available = lambda: True
    except Exception:
        pass

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    model_dirs = [
        "sinmate-gemma-3-1b-merged",
        "Sinmate/sinmate-gemma-3-1b-merged",
    ]
    m_path = "google/gemma-3-1b-it"
    for p in model_dirs:
        if os.path.exists(p):
            m_path = os.path.abspath(p)
            break

    # Gemma 3 overflows in fp16 (see Cell 4): bf16 on Ampere+, fp32 on the T4
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32

    print(f"Loading HuggingFace fallback from: {m_path} ...")
    hf_tokenizer = AutoTokenizer.from_pretrained(m_path)
    if hf_tokenizer.pad_token is None:
        hf_tokenizer.pad_token = hf_tokenizer.eos_token
    hf_model = AutoModelForCausalLM.from_pretrained(m_path, dtype=dtype, device_map="auto")
    hf_model.eval()

def chat_via_hf(user_text, persona_key="1", history=None):
    init_hf_fallback()
    import torch
    from transformers import TextStreamer

    persona_name = PERSONAS.get(persona_key, PERSONAS["1"])[0]
    messages = build_messages(user_text, persona_key, history)

    prompt = hf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = hf_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(hf_model.device)
    streamer = TextStreamer(hf_tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"\n[{persona_name}] Sinmate: ", end="")
    with torch.no_grad():
        outputs = hf_model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=64,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    return hf_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def chat_sinmate(user_text, persona_key="1", history=None):
    global ollama_available
    if ollama_available:
        try:
            return chat_via_ollama(user_text, persona_key, history)
        except urllib.error.URLError as e:  # HTTPError is a subclass
            print(f"\n⚠️ Ollama request failed - falling back to HuggingFace Transformers:\n{ollama_error(e)}")
            ollama_available = False
    return chat_via_hf(user_text, persona_key, history)

# 4. Print status & run automated multi-persona tests
print("=" * 60)
if ollama_available:
    print(f"🚀 SINMATE REAL-TIME CHAT (ENGINE: Ollama - {MODEL_TAG})")
else:
    print("🚀 SINMATE REAL-TIME CHAT (ENGINE: HuggingFace Transformers)")
print("=" * 60)

sample_tests = [
    ("Mata ada mara kammali ban, monada karanna hoda?", "1"),
    ("Mata ada godak mahansi darling.", "2"),
    ("Mata godak bayai exam eka gana.", "3"),
    ("Mage chocolate eka kawe oyada?", "4"),
]

for prompt_text, p_key in sample_tests:
    print(f"\nUser ({PERSONAS[p_key][0]}): {prompt_text}")
    chat_sinmate(prompt_text, persona_key=p_key)

print("\n" + "=" * 60)
print("💬 Interactive Real-Time Chat:")
print("Commands: 'p 1-4' to change persona | 'exit' to quit")
print("Personas: [1] Best Friend   [2] Romantic Partner   [3] Counsellor   [4] Playful Sibling")
print("=" * 60)

try:
    current_persona = "1"
    history = []
    while True:
        user_input = input("\nYou (or 'p 1-4' to switch, 'exit' to quit): ").strip()
        if not user_input or user_input.lower() in ["exit", "quit"]:
            print("Chat session ended. Sinmate signing off!")
            break
        if user_input.startswith("p ") and user_input[2:] in PERSONAS:
            current_persona = user_input[2:]
            history = []  # new persona starts a fresh conversation
            print(f"Switched persona to: {PERSONAS[current_persona][0]}")
            continue
        reply = chat_sinmate(user_input, persona_key=current_persona, history=history)
        history += [{"role": "user", "content": user_input}, {"role": "assistant", "content": reply}]
except (EOFError, KeyboardInterrupt):
    print("\nSession finished.")